# Galaxy-X-os  —  One-Click Colab Pipeline (SCALE x ODYSSEY)

Classify raw astronomical images into **5 celestial categories** with EfficientNet-B3.

**How to run:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.

Pipeline: clone → install → (optional Kaggle) → prepare data → train → evaluate → Grad-CAM → download results.

Every cell is idempotent — re-running is safe.

## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Srujan0798/Galaxy-X-os"
REPO_DIR = "/content/Galaxy-X-os"

# Detect if we are already inside the repo (e.g. mounted Drive); else clone.
if os.path.exists("src/prepare_data.py"):
    REPO_DIR = os.getcwd()
    print(f"Already inside repo at {REPO_DIR}")
elif os.path.exists(os.path.join(REPO_DIR, "src/prepare_data.py")):
    print(f"Repo already cloned at {REPO_DIR}")
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

# Core deps + extras needed for the real-first data pipeline.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "astroNN", "kagglehub", "kaggle", "h5py"], check=False)
print("Dependencies installed.")

## Cell 1b — Verify GPU is present (fail loudly if not)

Training EfficientNet-B3 on CPU is impractically slow. If this fails, set
`Runtime → Change runtime type → Hardware accelerator → GPU` and re-run.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "\n\n  NO GPU DETECTED.\n"
    "  Fix: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4),\n"
    "  then Runtime -> Run all.\n"
)
print("GPU OK:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)

## Cell 2 - (Optional) Kaggle token for REAL nebula / cluster / planetary

Spiral & elliptical always come from real **Galaxy10** (no key needed).

For real **nebula / star_cluster / planetary**, provide your Kaggle token in the next cell:

- **Easiest:** paste the `KGAT_...` string into the box in the next cell.
- **Or:** add it as a Colab Secret named `KAGGLE_API_TOKEN` (key icon in the sidebar) and leave the box empty.
- Get a new token at: https://www.kaggle.com/settings -> *Create New Token*.

**Skipping this** is fine - those three classes fall back to clearly-labelled procedural
images so the pipeline never breaks. The choice is recorded honestly in
`data/processed/DATA_MANIFEST.json`.

> Token stays in **your** Colab session only. Rotate it on Kaggle after submission.


In [ ]:
# =========================================================================
# OPTIONAL: Kaggle token -> enables REAL nebula / star_cluster / planetary
# =========================================================================
# Spiral + elliptical always come from real Galaxy10 (no key needed).
#
# Three ways to provide the new KGAT_ token (only ONE is required):
#
#   (A) Paste it in the box below.  Easiest.
#         -> Set KAGGLE_API_TOKEN = "KGAT_your_token_here"
#
#   (B) Colab Secret (left sidebar -> key icon):
#         -> Name: KAGGLE_API_TOKEN
#         -> Value: KGAT_your_token_here
#         -> Leave the box below empty.
#
#   (C) Legacy kaggle.json (upload via Files panel on the left).
#
# SECURITY: nothing you paste here leaves this Colab session. Do NOT commit.
# Rotate the token on Kaggle (Settings -> Create New Token) after submission.
# =========================================================================

import os
from pathlib import Path

# ---- (A) Paste your token between the quotes (or leave empty) --------------
KAGGLE_API_TOKEN = ""
# ----------------------------------------------------------------------------

# ---- (B) Auto-pull from a Colab Secret named KAGGLE_API_TOKEN --------------
if not KAGGLE_API_TOKEN:
    try:
        from google.colab import userdata
        KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN") or ""
    except Exception:
        pass
# ----------------------------------------------------------------------------

# ---- Install the token for the kagglehub / kaggle CLI ----------------------
if KAGGLE_API_TOKEN:
    token = KAGGLE_API_TOKEN.strip()
    os.environ["KAGGLE_API_TOKEN"] = token
    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)
    (kdir / "access_token").write_text(token + "\n")
    os.chmod(kdir / "access_token", 0o600)
    print("Kaggle token installed -> REAL nebula/cluster/planetary will be attempted.")
else:
    print("No Kaggle token -> procedural fallback for nebula/cluster/planetary.")
# ----------------------------------------------------------------------------

# ---- Status summary --------------------------------------------------------
has_creds = (
    bool(KAGGLE_API_TOKEN)
    or (Path.home() / ".kaggle" / "kaggle.json").exists()
)
print("Kaggle credentials present:", has_creds)
# ----------------------------------------------------------------------------


## Cell 3 — Prepare data (real-first, safe-fallback, idempotent)

Builds `data/processed/{train,val,test}/<class>/`, runs a disjoint 80/10/10
stratified split with an MD5 leakage check, and writes `DATA_MANIFEST.json`
+ `class_weights.json`.

In [ ]:
!python src/prepare_data.py --per-class 500

import json
with open("data/processed/DATA_MANIFEST.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Cell 4 — Train (full EfficientNet-B3 fine-tune on the GPU)

Writes `checkpoints/best_model.pth` (best val accuracy). Target: **> 80%** val accuracy.
Uses `configs/config.yaml` (progressive unfreezing, OneCycleLR, mixed precision, early stopping).

In [ ]:
!python src/train.py

import torch
ckpt = torch.load("checkpoints/best_model.pth", map_location="cpu", weights_only=True)
print(f"\nBest val accuracy: {ckpt['best_val_acc']:.4f}  (epoch {ckpt['epoch'] + 1})")

## Cell 5 — Evaluate (standard + Test-Time Augmentation)

Shows `results/evaluation_results.json` and the confusion matrix inline.

In [ ]:
!python src/evaluate.py

import json
from IPython.display import Image as IPyImage, display
with open("results/evaluation_results.json") as f:
    print(json.dumps(json.load(f), indent=2))
for p in ["results/confusion_matrix.png", "results/per_class_metrics.png"]:
    try:
        display(IPyImage(filename=p))
    except Exception as e:
        print(f"(could not display {p}: {e})")

## Cell 6 — Grad-CAM from the REAL trained model

In [ ]:
!python src/gradcam.py

import glob
from IPython.display import Image as IPyImage, display
cams = sorted(glob.glob("results/gradcam/*.png"))
print(f"Generated {len(cams)} Grad-CAM images.")
for p in cams[:4]:
    display(IPyImage(filename=p))

## Cell 7 — Zip results + checkpoint and download

After download: **unzip `results.zip` into the repo root, then commit.**

In [ ]:
import zipfile, os

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk("results"):
        for fn in fnames:
            fp = os.path.join(root, fn)
            z.write(fp, fp)
    if os.path.exists("checkpoints/best_model.pth"):
        z.write("checkpoints/best_model.pth", "checkpoints/best_model.pth")

print("Wrote results.zip:", round(os.path.getsize("results.zip") / 1e6, 1), "MB")
print("\nAfter download:\n  1. unzip results.zip into the repo root\n"
      "  2. git add results checkpoints && git commit -m 'Add trained model + results'")

try:
    from google.colab import files
    files.download("results.zip")
except Exception as e:
    print(f"(auto-download unavailable: {e} — download results.zip from the Files panel)")